<a href="https://colab.research.google.com/github/Rushikeshaya/Generative-AI/blob/main/practical4_GAN_CIFAR10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Imports and reproducibility
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Reshape, Flatten, Dropout
from tensorflow.keras.layers import Conv2D, Conv2DTranspose, LeakyReLU
from tensorflow.keras.optimizers import Adam

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))


TensorFlow: 2.21.0
GPU devices: []


In [ ]:
# 2. Load CIFAR-10 and prepare a smaller subset for faster training
(train_images, _), (_, _) = cifar10.load_data()

MAX_TRAIN_SAMPLES = 10000  # smaller subset to target the 40-minute limit
train_images = train_images[:MAX_TRAIN_SAMPLES].astype("float32")
train_images = (train_images - 127.5) / 127.5  # scale pixels to [-1, 1]

print("Training images:", train_images.shape)

# Preview a few real images
plt.figure(figsize=(6, 6))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.imshow(((train_images[i] + 1) * 127.5).astype("uint8"))
    plt.axis("off")
plt.tight_layout()
plt.show()


 26877952/170498071 ━━━━━━━━━━━━━━━━━━━━ 33:25 14us/step

In [ ]:
# 3. Define the discriminator
def define_discriminator(input_shape=(32, 32, 3)):
    model = Sequential([
        Conv2D(64, (3, 3), padding="same", input_shape=input_shape),
        LeakyReLU(negative_slope=0.2),

        Conv2D(128, (3, 3), strides=(2, 2), padding="same"),
        LeakyReLU(negative_slope=0.2),

        Conv2D(128, (3, 3), strides=(2, 2), padding="same"),
        LeakyReLU(negative_slope=0.2),

        Conv2D(256, (3, 3), strides=(2, 2), padding="same"),
        LeakyReLU(negative_slope=0.2),

        Flatten(),
        Dropout(0.4),
        Dense(1, activation="sigmoid")
    ])
    model.compile(
        loss="binary_crossentropy",
        optimizer=Adam(learning_rate=0.0002, beta_1=0.5),
        metrics=["accuracy"]
    )
    return model

discriminator = define_discriminator()
discriminator.summary()


In [ ]:
# 4. Define the generator
LATENT_DIM = 100

def define_generator(latent_dim=LATENT_DIM):
    model = Sequential([
        Dense(256 * 4 * 4, input_shape=(latent_dim,)),
        LeakyReLU(negative_slope=0.2),
        Reshape((4, 4, 256)),

        Conv2DTranspose(128, (4, 4), strides=(2, 2), padding="same"),
        LeakyReLU(negative_slope=0.2),

        Conv2DTranspose(128, (4, 4), strides=(2, 2), padding="same"),
        LeakyReLU(negative_slope=0.2),

        Conv2DTranspose(128, (4, 4), strides=(2, 2), padding="same"),
        LeakyReLU(negative_slope=0.2),

        Conv2D(3, (3, 3), activation="tanh", padding="same")
    ])
    return model

generator = define_generator()
generator.summary()


In [ ]:
# 5. Helpers and combined GAN
def generate_latent_points(latent_dim, n_samples):
    return np.random.randn(n_samples, latent_dim).astype("float32")

def generate_real_samples(dataset, n_samples):
    indices = np.random.randint(0, dataset.shape[0], n_samples)
    images = dataset[indices]
    labels = np.ones((n_samples, 1), dtype="float32")
    return images, labels

def generate_fake_samples(g_model, latent_dim, n_samples):
    latent_points = generate_latent_points(latent_dim, n_samples)
    # Calling the model directly avoids the overhead of predict() in every batch.
    images = g_model(latent_points, training=False).numpy()
    labels = np.zeros((n_samples, 1), dtype="float32")
    return images, labels

def define_gan(g_model, d_model):
    d_model.trainable = False
    model = Sequential([g_model, d_model])
    model.compile(
        loss="binary_crossentropy",
        optimizer=Adam(learning_rate=0.0002, beta_1=0.5)
    )
    return model

gan = define_gan(generator, discriminator)


In [ ]:
# 6. Plot generated images after each epoch
def save_generated_images(g_model, epoch, latent_dim=LATENT_DIM, n=25):
    latent_points = generate_latent_points(latent_dim, n)
    images = g_model(latent_points, training=False).numpy()
    images = np.clip((images + 1.0) / 2.0, 0, 1)

    plt.figure(figsize=(7, 7))
    for i in range(n):
        plt.subplot(5, 5, i + 1)
        plt.imshow(images[i])
        plt.axis("off")
    plt.suptitle(f"Generated images after epoch {epoch}")
    plt.tight_layout()
    filename = f"cifar10_generated_epoch_{epoch:02d}.png"
    plt.savefig(filename, dpi=120)
    plt.show()
    plt.close()
    print("Saved:", filename)


In [ ]:
# 7. Train for exactly 10 epochs
EPOCHS = 10
BATCH_SIZE = 128
STEPS_PER_EPOCH = len(train_images) // BATCH_SIZE

start_time = time.time()
history = {"d_real_loss": [], "d_fake_loss": [], "g_loss": []}

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    d_real_losses, d_fake_losses, g_losses = [], [], []

    for step in range(STEPS_PER_EPOCH):
        half_batch = BATCH_SIZE // 2

        # Train discriminator on real images
        real_images, real_labels = generate_real_samples(train_images, half_batch)
        d_real_loss, _ = discriminator.train_on_batch(real_images, real_labels)

        # Train discriminator on generated images
        fake_images, fake_labels = generate_fake_samples(generator, LATENT_DIM, half_batch)
        d_fake_loss, _ = discriminator.train_on_batch(fake_images, fake_labels)

        # Train generator through the combined GAN
        latent_points = generate_latent_points(LATENT_DIM, BATCH_SIZE)
        misleading_labels = np.ones((BATCH_SIZE, 1), dtype="float32")
        g_loss = gan.train_on_batch(latent_points, misleading_labels)

        # Keras may return a scalar or a list, depending on version/configuration.
        d_real_losses.append(float(np.asarray(d_real_loss).reshape(-1)[0]))
        d_fake_losses.append(float(np.asarray(d_fake_loss).reshape(-1)[0]))
        g_losses.append(float(np.asarray(g_loss).reshape(-1)[0]))

    history["d_real_loss"].append(float(np.mean(d_real_losses)))
    history["d_fake_loss"].append(float(np.mean(d_fake_losses)))
    history["g_loss"].append(float(np.mean(g_losses)))

    elapsed = (time.time() - epoch_start) / 60
    total_elapsed = (time.time() - start_time) / 60
    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"D(real): {history['d_real_loss'][-1]:.4f} | "
        f"D(fake): {history['d_fake_loss'][-1]:.4f} | "
        f"G: {history['g_loss'][-1]:.4f} | "
        f"epoch: {elapsed:.1f} min | total: {total_elapsed:.1f} min"
    )

    save_generated_images(generator, epoch)

generator.save("cifar10_dcgan_generator.keras")
print(f"Training finished in {(time.time() - start_time) / 60:.1f} minutes.")
print("Saved generator: cifar10_dcgan_generator.keras")


In [ ]:
# 8. Plot training losses
plt.figure(figsize=(8, 4))
plt.plot(range(1, EPOCHS + 1), history["d_real_loss"], marker="o", label="Discriminator: real")
plt.plot(range(1, EPOCHS + 1), history["d_fake_loss"], marker="o", label="Discriminator: fake")
plt.plot(range(1, EPOCHS + 1), history["g_loss"], marker="o", label="Generator")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("DCGAN training losses on CIFAR-10")
plt.xticks(range(1, EPOCHS + 1))
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()
